# Phase 3 Real Acoustic Feature-Based Bayesian Multilevel Model

This notebook fits the final empirical acoustic feature model on the cleaned real participant ratings. It does not modify raw data, cleaning decisions, the pre-specified primary model, or frozen acoustic feature definitions.

Primary Gaussian model:

`rating ~ episode + group + z_RMS + z_CF + z_SW + (1 | participant_id) + (1 | stimulus_id)`

SI sensitivity Gaussian model:

`rating ~ episode + group + z_RMS + z_CF + z_SW + z_SI + (1 | participant_id) + (1 | stimulus_id)`

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif not (PROJECT_ROOT / "statistical-baseline").exists():
    PROJECT_ROOT = next(path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / "statistical-baseline").exists())

sys.path.insert(0, str(PROJECT_ROOT / "statistical-baseline" / "src"))

from statistical_baseline.real_feature_model import (
    BOUNDED_FORMULA,
    OUTPUT_DIR,
    PRIMARY_FORMULA,
    RATINGS_PATH,
    SI_FORMULA,
    run_real_feature_model,
)

RATINGS_PATH, OUTPUT_DIR, PRIMARY_FORMULA, SI_FORMULA, BOUNDED_FORMULA

WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`


WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


(WindowsPath('C:/Users/oscar/Documents/7. QMUL UNIVERSITY/1. Master Program/3. MSc Project/intent2control-dissertation/statistical-baseline/data/real/real_ratings_clean.csv'),
 WindowsPath('C:/Users/oscar/Documents/7. QMUL UNIVERSITY/1. Master Program/3. MSc Project/intent2control-dissertation/statistical-baseline/outputs/real_feature_model'),
 'rating ~ episode + group + z_RMS + z_CF + z_SW + (1 | participant_id) + (1 | stimulus_id)',
 'rating ~ episode + group + z_RMS + z_CF + z_SW + z_SI + (1 | participant_id) + (1 | stimulus_id)',
 'rating_01 ~ episode + group + z_RMS + z_CF + z_SW + (1 | participant_id) + (1 | stimulus_id)')

## Execute Locked Empirical Feature-Model Pipeline

The Gaussian primary and SI sensitivity models use Bambi/PyMC, Gaussian likelihood, 4 chains, 1000 tuning draws, 1000 posterior draws, `target_accept=0.95`, deterministic seed, and nutpie. The bounded sensitivity model uses PyMC Beta regression after a documented transformation away from exact 0/1 values because the empirical rating outcome contains exact 0 and 100 responses.

In [2]:
outputs = run_real_feature_model()
outputs.keys()

C:\Users\oscar\miniconda3\lib\site-packages\pymc\sampling\mcmc.py:328: UserWarning: `idata_kwargs` are currently ignored by the nutpie sampler
  warnings.warn(


Progress,Draws,Divergences,Step Size,Gradients/Draw
,2000,0,0.20,31
,2000,0,0.19,31
,2000,0,0.20,31
,2000,0,0.20,31


C:\Users\oscar\miniconda3\lib\site-packages\pymc\sampling\mcmc.py:328: UserWarning: `idata_kwargs` are currently ignored by the nutpie sampler
  warnings.warn(


Progress,Draws,Divergences,Step Size,Gradients/Draw
,2000,0,0.19,31
,2000,0,0.19,31
,2000,0,0.18,31
,2000,0,0.18,31


C:\Users\oscar\miniconda3\lib\site-packages\pymc\sampling\mcmc.py:328: UserWarning: `idata_kwargs` are currently ignored by the nutpie sampler
  warnings.warn(


C:\Users\oscar\miniconda3\lib\site-packages\pytensor\tensor\rewriting\elemwise.py:954: UserWarning: Loop fusion failed because the resulting node would exceed the kernel argument limit.
  warn(


Progress,Draws,Divergences,Step Size,Gradients/Draw
,2000,0,0.19,31
,2000,0,0.19,63
,2000,0,0.19,31
,2000,0,0.19,31


Sampling: [rating_01]


C:\Users\oscar\miniconda3\lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

dict_keys(['validation', 'descriptive', 'primary_idata', 'si_idata', 'bounded_idata', 'primary_fixed', 'si_fixed', 'bounded_summary', 'variance', 'icc', 'ppc', 'bounded_ppc', 'diagnostics', 'loo', 'stimulus_feature_loo', 'expected', 'winners', 'winner_validation', 'observed_vs_model', 'comparison', 'runtimes', 'bounded_status', 'findings'])

## Dataset and Feature Mapping Validation

In [3]:
outputs["validation"]

,check,passed,actual_value
0,final_analysable_n_30,True,30
1,group_01_n_16,True,16
2,group_02_n_14,True,14
3,rating_observations_900,True,900
4,stimuli_20,True,20
5,songs_4,True,4
6,episodes_3,True,3
7,ratings_per_participant_30,True,"{""count"": 30.0, ""mean"": 30.0, ""std"": 0.0, ""min..."
8,no_missing_outcome,True,0
9,no_invalid_ratings,True,0


## Descriptive Feature Summaries

These are descriptive only. Acoustic variation exists at 20 unique stimulus profiles, not 900 independent rating observations.

In [4]:
outputs["descriptive"]

,feature,quartile,n,rating_mean,rating_median,rating_sd,rating_min,rating_max,unique_stimuli,interpretation_note
0,z_RMS,Q1,264,46.045455,48.0,32.885298,0.0,100.0,6,descriptive only; acoustic predictors vary at ...
1,z_RMS,Q2,186,44.005376,46.0,30.967725,0.0,100.0,4,descriptive only; acoustic predictors vary at ...
2,z_RMS,Q3,270,59.033333,64.5,31.492886,0.0,100.0,6,descriptive only; acoustic predictors vary at ...
3,z_RMS,Q4,180,64.533333,71.5,27.779338,0.0,100.0,4,descriptive only; acoustic predictors vary at ...
4,z_CF,Q1,234,55.072650,57.0,30.393398,0.0,100.0,5,descriptive only; acoustic predictors vary at ...
5,z_CF,Q2,228,53.578947,59.5,32.245177,0.0,100.0,5,descriptive only; acoustic predictors vary at ...
6,z_CF,Q3,222,56.103604,58.0,31.297768,0.0,100.0,5,descriptive only; acoustic predictors vary at ...
7,z_CF,Q4,216,47.861111,49.0,34.214497,0.0,100.0,5,descriptive only; acoustic predictors vary at ...
8,z_SW,Q1,228,56.122807,60.5,32.233493,0.0,100.0,5,descriptive only; acoustic predictors vary at ...
9,z_SW,Q2,234,53.615385,58.0,31.249335,0.0,100.0,5,descriptive only; acoustic predictors vary at ...


## Fixed Effects

In [5]:
outputs["primary_fixed"], outputs["si_fixed"], outputs["bounded_summary"]

(        model_name             term    mean     sd   hdi_3  hdi_97   mcse  \
 0  primary_feature        Intercept  45.893  5.353  35.927  56.151  0.183   
 1  primary_feature   episode[EDR-2]   2.163  2.258  -2.159   6.254  0.035   
 2  primary_feature    episode[FM-1]  -6.303  2.256 -10.509  -2.051  0.034   
 3  primary_feature  group[group_02]  17.669  7.744   2.167  31.249  0.254   
 4  primary_feature            z_RMS   5.508  2.753  -0.026  10.459  0.085   
 5  primary_feature             z_CF  -6.195  3.003 -11.657  -0.259  0.084   
 6  primary_feature             z_SW  -3.165  2.895  -8.677   2.177  0.084   
 
    mcse_sd  ess_bulk  ess_tail  r_hat  
 0    0.120     864.0    1112.0    1.0  
 1    0.034    4271.0    3008.0    1.0  
 2    0.033    4446.0    3345.0    1.0  
 3    0.154     930.0    1372.0    1.0  
 4    0.079    1108.0    1206.0    1.0  
 5    0.071    1265.0    1343.0    1.0  
 6    0.057    1177.0    1911.0    1.0  ,
        model_name             term    mean  

## ICCs and Variance Components

In [6]:
outputs["variance"], outputs["icc"]

(        model_name                    term    mean     sd   hdi_3  hdi_97  \
 0  primary_feature  1|participant_id_sigma  11.644  1.987   8.158  15.346   
 1  primary_feature     1|stimulus_id_sigma  10.714  2.534   6.499  15.647   
 2  primary_feature                   sigma  27.474  0.680  26.161  28.713   
 3   si_sensitivity  1|participant_id_sigma  11.675  1.992   8.214  15.435   
 4   si_sensitivity     1|stimulus_id_sigma  11.316  2.874   6.846  16.663   
 5   si_sensitivity                   sigma  27.452  0.662  26.217  28.647   
 
     mcse  mcse_sd  ess_bulk  ess_tail  r_hat  
 0  0.063    0.039    1010.0    1560.0    1.0  
 1  0.076    0.063    1127.0    1662.0    1.0  
 2  0.010    0.010    4977.0    3214.0    1.0  
 3  0.056    0.043    1279.0    1875.0    1.0  
 4  0.087    0.110    1201.0    1728.0    1.0  
 5  0.009    0.011    5199.0    2821.0    1.0  ,
          model_name                  term     mean      sd    hdi_3   hdi_97  \
 0   primary_feature  participant_

## Convergence Diagnostics

In [7]:
outputs["diagnostics"]

,model_name,divergences,max_rhat,min_bulk_ess,min_tail_ess,max_tree_depth,tree_depth_warnings,energy_warnings,runtime_seconds,sampler
0,primary_feature,0,1.00,864.0,1112.0,NaN,0,0,30.696777,bambi_pymc_nutpie
1,si_sensitivity,0,1.00,1201.0,1675.0,NaN,0,0,22.926781,bambi_pymc_nutpie
2,bounded_beta_primary_sensitivity,0,1.01,942.0,1366.0,NaN,0,0,61.752559,explicit_pymc_beta_nutpie


## Posterior Predictive Checks and Bounded Sensitivity

In [8]:
outputs["ppc"], outputs["bounded_status"]

(                          model_name                              metric  \
 0                    primary_feature                       observed_mean   
 1                    primary_feature                         observed_sd   
 2                    primary_feature      posterior_predictive_mean_mean   
 3                    primary_feature        posterior_predictive_sd_mean   
 4                    primary_feature        posterior_predictive_below_0   
 5                    primary_feature      posterior_predictive_above_100   
 6                    primary_feature  posterior_predictive_outside_0_100   
 7                    primary_feature         observed_episode_mean_EDR-1   
 8                    primary_feature              ppc_episode_mean_EDR-1   
 9                    primary_feature         observed_episode_mean_EDR-2   
 10                   primary_feature              ppc_episode_mean_EDR-2   
 11                   primary_feature          observed_episode_mean_FM-1   

## PSIS-LOO Comparisons

In [9]:
outputs["loo"], outputs["stimulus_feature_loo"]

(        model_name     elpd_loo         se      p_loo  pareto_k_gt_0_7  \
 0  primary_feature -4281.366263  17.152025  44.166449                0   
 1   si_sensitivity -4280.684938  17.156354  43.754136                0   
 
    pareto_k_max  loo_valid  elpd_diff_from_best  
 0      0.304638       True            -0.681325  
 1      0.334555       True             0.000000  ,
         model_name     elpd_loo         se      p_loo  pareto_k_gt_0_7  \
 0   stimulus_model -4281.126621  17.130499  44.188772                0   
 1  primary_feature -4281.366263  17.152025  44.166449                0   
 
    pareto_k_max  loo_valid  elpd_diff_from_best  
 0      0.307396       True             0.000000  
 1      0.304638       True            -0.239642  )

## Posterior Expected Ratings and Preferred-Mix Probabilities

Posterior preferred-mix probabilities are calculated draw by draw within each `song_id x episode` combination, with fractional credit for exact draw-level ties. Because the primary model has no Episode x Feature interaction, relative acoustic-feature contributions do not vary by context in this model.

In [10]:
outputs["expected"].head(), outputs["winners"].head(20), outputs["winner_validation"]

(        model_name     group episode          song_id             stimulus_id  \
 0  primary_feature  group_01   EDR-1  id_like_to_know  id_like_to_know_pxl_s1   
 1  primary_feature  group_01   EDR-1  id_like_to_know  id_like_to_know_pxl_s2   
 2  primary_feature  group_01   EDR-1  id_like_to_know  id_like_to_know_pxl_s3   
 3  primary_feature  group_01   EDR-1  id_like_to_know  id_like_to_know_pxl_s5   
 4  primary_feature  group_01   EDR-1  id_like_to_know  id_like_to_know_pxl_s7   
 
              mix_id     z_RMS      z_CF      z_SW       mean     median  \
 0  mix_9ffcd672af17 -0.565911 -0.579304 -0.485121  50.184072  50.260047   
 1  mix_6f8b655d7999  0.533000 -1.187467 -0.763752  61.899785  61.988751   
 2  mix_65f41f41d3ab  0.323484 -1.065060 -1.165177  57.087071  57.203831   
 3  mix_82d60d7c7fc5 -0.358338 -0.152766  0.797271  44.442233  44.423633   
 4  mix_b393d3919c78 -0.867056 -0.618841 -0.322901  51.269289  51.326401   
 
          sd      hdi_3     hdi_97  probability_

## Observed vs Model Preferences

This is labelled in-sample descriptive agreement only. Proper held-out predictive comparison belongs to the later RQ4 workflow.

In [11]:
outputs["observed_vs_model"].head(40)

,song_id,episode,stimulus_id,observed_credit,observed_winner_rows,observed_tie_rows,human_trials,observed_share_fractional_ties,posterior_probability_highest,is_final_predicted_winner,model_winner_stimulus_id,observed_top_stimulus_id,in_sample_descriptive_agreement,comparison_scope
1,id_like_to_know,EDR-1,id_like_to_know_pxl_s2,2.000000,2.0,0.0,16.0,0.125000,0.81150,True,id_like_to_know_pxl_s2,id_like_to_know_pxl_s1,False,in-sample descriptive agreement; not out-of-sa...
2,id_like_to_know,EDR-1,id_like_to_know_pxl_s3,5.000000,5.0,0.0,16.0,0.312500,0.17375,False,id_like_to_know_pxl_s2,id_like_to_know_pxl_s1,False,in-sample descriptive agreement; not out-of-sa...
4,id_like_to_know,EDR-1,id_like_to_know_pxl_s7,3.000000,3.0,0.0,16.0,0.187500,0.01075,False,id_like_to_know_pxl_s2,id_like_to_know_pxl_s1,False,in-sample descriptive agreement; not out-of-sa...
0,id_like_to_know,EDR-1,id_like_to_know_pxl_s1,6.000000,6.0,0.0,16.0,0.375000,0.00400,False,id_like_to_know_pxl_s2,id_like_to_know_pxl_s1,False,in-sample descriptive agreement; not out-of-sa...
3,id_like_to_know,EDR-1,id_like_to_know_pxl_s5,0.000000,0.0,0.0,NaN,0.000000,0.00000,False,id_like_to_know_pxl_s2,id_like_to_know_pxl_s1,False,in-sample descriptive agreement; not out-of-sa...
6,id_like_to_know,EDR-2,id_like_to_know_pxl_s2,5.700000,7.0,2.0,16.0,0.356250,0.81150,True,id_like_to_know_pxl_s2,id_like_to_know_pxl_s2,True,in-sample descriptive agreement; not out-of-sa...
7,id_like_to_know,EDR-2,id_like_to_know_pxl_s3,4.700000,6.0,2.0,16.0,0.293750,0.17375,False,id_like_to_know_pxl_s2,id_like_to_know_pxl_s2,True,in-sample descriptive agreement; not out-of-sa...
9,id_like_to_know,EDR-2,id_like_to_know_pxl_s7,1.700000,3.0,2.0,16.0,0.106250,0.01075,False,id_like_to_know_pxl_s2,id_like_to_know_pxl_s2,True,in-sample descriptive agreement; not out-of-sa...
5,id_like_to_know,EDR-2,id_like_to_know_pxl_s1,2.700000,4.0,2.0,16.0,0.168750,0.00400,False,id_like_to_know_pxl_s2,id_like_to_know_pxl_s2,True,in-sample descriptive agreement; not out-of-sa...
8,id_like_to_know,EDR-2,id_like_to_know_pxl_s5,1.200000,2.0,1.0,16.0,0.075000,0.00000,False,id_like_to_know_pxl_s2,id_like_to_know_pxl_s2,True,in-sample descriptive agreement; not out-of-sa...


## Stimulus vs Feature Model Comparison

In [12]:
outputs["comparison"]

,aspect,stimulus_model,feature_model
0,formula,rating ~ episode + group + (1 | participant_id...,rating ~ episode + group + z_RMS + z_CF + z_SW...
1,runtime_seconds,20.077196,30.696777
2,participant_ICC_mean,0.124,0.136
3,stimulus_ICC_mean,0.177,0.117
4,ppc_outside_0_100,0.123635,0.124634
5,episode_EDR_2_mean,2.147,2.163
6,episode_FM_1_mean,-6.292,-6.303
7,group_group_02_mean,6.868,17.669
8,interpretability,direct stimulus-level baseline; does not expla...,native rating-point acoustic associations for ...
9,main_limitation,additive; no Episode x Stimulus interaction,features vary across only 20 stimuli; additive...


## RQ Relevance

RQ2: this model contributes evidence about listener/context variability through participant ICC, episode fixed effects, residual stimulus variance, and observed preference distributions.

RQ4: the fitted feature model forms a statistical baseline for later comparison with LLM preference predictions, but this notebook does not perform held-out LLM evaluation.

RQ5: the model estimates whether RMS, crest factor, and stereo width contain useful predictive information for human mix ratings. RQ5's primary evidence will ultimately also draw on broader LLM ablation analyses, not regression coefficients alone.

## Achieved Sample-Size Limitation

The planned target was 50 analysable participants; achieved N is 30 with group split 16 / 14. The lower-than-planned sample reduces precision, especially because acoustic predictors vary across only 20 unique stimuli. The planned model and target are not retroactively changed.

## Empirical Feature-Model Findings

In [13]:
print(outputs["findings"])
print("\nExport directory:", OUTPUT_DIR)

# Empirical Feature-Model Findings

The achieved analysable sample was N=30, with group_01=16 and group_02=14. The planned preferred analysable sample was N=50.
RMS: posterior mean 5.51 rating points per one-SD increase, 94% HDI [-0.03, 10.46].
Crest factor: posterior mean -6.20 rating points per one-SD increase, 94% HDI [-11.66, -0.26].
Stereo width: posterior mean -3.17 rating points per one-SD increase, 94% HDI [-8.68, 2.18].
SI sensitivity: posterior mean -1.64, 94% HDI [-7.78, 4.52]. SI remains sensitivity-only.
Acoustic coefficients are conditional associations in native rating points for the Gaussian models and should not be read causally.
Primary participant ICC mean 0.136, 94% HDI [0.070, 0.214]. Primary stimulus ICC mean 0.117, 94% HDI [0.041, 0.212].
Primary model diagnostics: divergences=0, max R-hat=1.000, min bulk ESS=864.0, min tail ESS=1112.0.
Gaussian PPC outside 0-100 was 0.1246; predictions were not clipped.
Bounded sensitivity status: fitted transformed-Beta sensiti